# Georgia Power ingestion

Run this notebook to produce `df_georgia` (all ITS sponsors) and `df_gpc` (GPC only). It combines the former PDF preparation and project-list notebooks. The original ZIP, extracted Volume 3 and selected PDFs are reused and preserved.

The functions in `gridlock.georgia_ingestion` reuse the original extraction code: ZIP validation, bookmark/contents checks, table cleanup, description matching and strategic summaries. No new dependencies are needed.

In [1]:
from gridlock.georgia_ingestion import (
    prepare_project_pdfs, read_project_list, read_descriptions,
    read_strategic_summaries, join_project_details, save_project_tables,
)

## Prepare verified PDF sections

Read the table of contents and printed labels before selecting pages. Existing working files are checked against their original section and reused. This is one step in ingestion, not a separate notebook dependency.

In [2]:
sections = prepare_project_pdfs()
sections[["section", "plan_first", "plan_last"]]

Reusing original ZIP: C:\Users\sebas\PycharmProjects\Git\Project_secret_project\data\raw\georgia_power\2025_irp\2025_irp_public_disclosure.zip
Volume 3: C:\Users\sebas\PycharmProjects\Git\Project_secret_project\data\raw\georgia_power\2025_irp\files\2025 IRP Volume 3 PUBLIC DISCLOSURE.pdf


Plan starts at Volume 3 PDF page: 171
Pages in the plan: 304
Table of contents found on Volume 3 PDF pages: [172, 173]
Georgia ITS 10 Year Expansion Plan Projects List | Plan pages: 7 - 20 | Volume 3 PDF pages: 177 - 190
Stability Project Details | Plan pages: 43 - 53 | Volume 3 PDF pages: 213 - 223
Short Circuit Project Details | Plan pages: 54 - 55 | Volume 3 PDF pages: 224 - 225
Interface Transfer Capability Project Details | Plan pages: 56 - 59 | Volume 3 PDF pages: 226 - 229
Steady State Project Details | Plan pages: 60 - 255 | Volume 3 PDF pages: 230 - 425
Strategic Projects | Plan pages: 272 - 297 | Volume 3 PDF pages: 442 - 467


,section,plan_first,plan_last
0,Georgia ITS 10 Year Expansion Plan Projects List,7,20
1,Stability Project Details,43,53
2,Short Circuit Project Details,54,55
3,Interface Transfer Capability Project Details,56,59
4,Steady State Project Details,60,255
5,Strategic Projects,272,297


## Extract projects and descriptions

A TEAMS number identifies a project. Wrapped table names are joined, and the detailed descriptions and strategic summaries are matched by that ID.

In [3]:
df_project_list = read_project_list()
df_descriptions = read_descriptions()
df_strategic = read_strategic_summaries()
df_georgia, df_gpc = join_project_details(df_project_list, df_descriptions, df_strategic)

Consider using the pymupdf_layout package for a greatly improved page layout analysis.


List IDs without descriptions: []
Description IDs outside the list: []
Strategic IDs outside the list: []


## Validate and save

The need date is preserved as `need_date`; it is not a confirmed in-service or construction date. The full ITS table remains available alongside the GPC-only table. Source pages and descriptions remain attached.

In [4]:
assert len(df_georgia) == 208
assert len(df_gpc) == 122
assert df_georgia.project_id.is_unique
assert df_georgia.description.notna().all()
output_dir = save_project_tables(df_georgia, df_gpc)
print(f"ITS projects: {len(df_georgia)}")
print(f"GPC projects: {len(df_gpc)}")
print(f"Saved to: {output_dir}")
df_gpc[["project_id", "project_name", "need_date", "description"]].head()

ITS projects: 208
GPC projects: 122
Saved to: C:\Users\sebas\PycharmProjects\Git\Project_secret_project\data\processed\georgia_power\2025_irp


,project_id,project_name,need_date,description
2,18492,MITCHELL - NORTH TIFTON 230KV RECONDUCTOR,2025-05-01,Rebuild 35.21 miles of the Mitchell - North Ti...
3,19676,ADAMSVILLE - JACK MCDONOUGH 230KV LINE REBUILD,2025-06-01,Rebuild the entire 6.1 mile Adamsville - Jack ...
4,18800,ECHECONNEE-WELLSTON 115KV REBUILD,2025-06-01,Rebuild the South Warner Robins to Wellston se...
5,19187,GRID - BREMEN - CROOKED CREEK (APC) 115 KV PRO...,2025-06-01,"Rebuild from Bremen to Indian Creek Road, on t..."
9,11821,JESUP - LUDOWICI PRIMARY 115KV REBUILD,2025-06-01,Rebuild approximately 7.5 miles of the Jesup -...
